# Train WEPR end to end, on any QA dataset

The same seven steps as the [ECIR
reproduction](https://github.com/artefactory/artefactual/tree/main/scripts/ecir), with one
difference in aim. That pipeline exists to reproduce the paper's numbers, so it fixes the
dataset, the prompts and the sampling, and runs both LLM stages as `vllm run-batch` jobs
on a GPU box. This notebook exists to get **a detector for your model, from your data**,
so it runs the same stages against any OpenAI-compatible endpoint and takes whatever
short-form QA set you point it at.

| | ECIR pipeline | This notebook |
|---|---|---|
| Goal | Reproduce the paper | Train on your own data |
| Dataset | TriviaQA, fixed | Any QA set with short answers |
| Runs on | A Linux GPU box | Anything with an endpoint |
| Both LLM stages | `vllm run-batch` | Your endpoint |
| Prompts | The paper's, verbatim | The paper's, verbatim |

The steps, and which of them costs anything:

| Phase | Steps | Cost |
|---|---|---|
| **A. Prepare** | 1. Build the question pack | seconds |
| **B. Run the model** | 2. Generate answers with logprobs · 3. Judge them | the whole cost |
| **C. Train** | 4. Fit · 5. Evaluate · 6. Save · 7. Use | seconds, rerun freely |

Phase C never needs phase B repeated, so retraining at a different `k` or on `epr` instead
is free once the answers are labelled. Saving phase B's two outputs is worth it for that
reason alone — and if you only want to see phase C, {doc}`train_wepr` runs it on a shipped
sample with no endpoint at all.

## Prerequisites

```bash
uv pip install "artefactual[adapters]" datasets
```

| Variable | Required | What it is |
|---|---|---|
| `OPENAI_BASE_URL` | yes | Any OpenAI-compatible endpoint returning `top_logprobs` |
| `OPENAI_API_KEY` | yes | Its key |
| `OPENAI_MODEL` | no | The model being scored — the detector you train belongs to it |

**A detector belongs to the model it was trained on.** Its weights read that model's
confidence, so scoring a different one with them is not supported; retrain instead. This
notebook is not executed when the docs are built, because it generates against a live
endpoint — the numbers you see are the ones your run produces.

In [ ]:
import json
import os

import numpy as np

MODEL = os.environ.get("OPENAI_MODEL", "mistralai/Ministral-8B-Instruct-2410")

# Ranks kept per token. Part of the feature definition, not a batch size: it must be the
# same value in step 2 and step 4, and mismatching them is the usual way to end up with a
# detector that scores wrongly. Every published detector uses 15.
K = 15

N_QUESTIONS = 300
SEED = 42

## Step 1 — build the question pack

The pack is a list of `{question, question_id, short_answer, answer_aliases}` — the same
schema `scripts/ecir` reads, so anything you build here also feeds the batch pipeline.

TriviaQA's closed-book configuration carries all four fields, so the mapping is a rename.
Two properties decide whether another dataset works, and neither is about its columns:
answers must be **short enough for a judge to grade**, and the model must get **enough of
them wrong** that both classes appear. A model that answers everything correctly gives the
fit nothing to learn from.

Shuffled before slicing, because splits arrive grouped by source and the head of one is a
narrower sample than the same count drawn at random.

In [ ]:
from datasets import load_dataset

rows = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")

questions = [
    {
        "question": row["question"],
        "question_id": row["question_id"],
        "short_answer": row["answer"]["value"],
        # `value` is not always among `aliases`, so it is added rather than assumed present.
        "answer_aliases": [a for a in row["answer"]["aliases"] if a.casefold() != row["answer"]["value"].casefold()],
    }
    for row in rows.shuffle(seed=SEED).select(range(N_QUESTIONS))
]

# question_id joins every later stage, exactly as custom_id does in the batch pipeline, so
# a duplicate would pair an answer with another question's verdict.
assert len({q["question_id"] for q in questions}) == len(questions), "question_id is not unique"
print(f"{len(questions)} questions, e.g. {questions[0]['question']!r} -> {questions[0]['short_answer']!r}")

## Step 2 — generate the answers, keeping the log-probabilities

`logprobs=True` and `top_logprobs=K` are what make a response scoreable at all: the
detector reads the token distribution behind the answer, not the answer. A response
generated without them carries nothing to score, and one generated with fewer than `K`
ranks is refused when it reaches the parser rather than quietly zero-filled — so
generating *wider* than you need is safe, and narrower costs another run.

Prompt and sampling are the paper's (§4.1.2): non-greedy at `T = 1.0`, `top_p = 1.0`.
Non-greedy is the point — the method measures hesitation in the raw distribution.

Requests are threaded because the round trips, not the fitting, are what make this notebook
slow. A failure returns `None` and is dropped together with its question, so a rate-limited
request costs one example rather than the run.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI

client = OpenAI()  # reads OPENAI_BASE_URL and OPENAI_API_KEY

GENERATE = """You are a useful assistant that help finding short and precise answers for a given query or question.
            Please keep your output AS SHORT AND CONCISE AS POSSIBLE.
            Here is the query :
            {query}
            """


def generate(question):
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": GENERATE.format(query=question["question"])}],
            logprobs=True,
            top_logprobs=K,
            temperature=1.0,
            top_p=1.0,
            max_completion_tokens=200,
        )
    except Exception as error:  # noqa: BLE001 - one failed request should not cost the run
        print(f"dropped {question['question'][:60]!r}: {error}")
        return None


with ThreadPoolExecutor(max_workers=8) as pool:
    generated = list(pool.map(generate, questions))

# `map` preserves order, so a question and its answer stay paired through the filter.
paired = [(q, r) for q, r in zip(questions, generated) if r is not None]
questions, responses = [q for q, _ in paired], [r for _, r in paired]
print(f"{len(responses)}/{len(generated)} generated")

In [ ]:
# The cheapest place to notice an endpoint that ignored `top_logprobs`: it returns a
# perfectly valid completion carrying nothing to score.
width = len(responses[0].choices[0].logprobs.content[0].top_logprobs)
assert width >= K, f"endpoint returned {width} ranks per token, need {K}"

print(f"{width} ranks per token")
print(responses[0].choices[0].message.content)

## Step 3 — judge the answers

The paper's judge prompt, verbatim, from `scripts/ecir/prompts/judge.txt`. The judge sees
the question, the gold answer, its aliases and the generated answer, and replies with
`{"judgment": true|false, "explanation": "..."}`.

**`judgment: true` means the answer was correct**, so the training label is its negation:
`1` marks a hallucination. Getting that backwards trains a detector that is exactly wrong,
which is why it is spelled out here and in the batch pipeline both.

Grading is deterministic — `temperature=0` — and a verdict that will not parse is dropped
with its example rather than guessed at.

In [ ]:
# The paper's judge prompt, read verbatim from scripts/ecir/prompts/judge.txt. Rendered by
# literal replacement rather than str.format, for the same reason the shell pipeline uses
# jq's split/join: the prompt ends with a JSON example, whose braces `format` would try to
# read as fields, and a question containing braces would corrupt the rest.
JUDGE = """You are an expert evaluator tasked with determining if two answers convey compatible information. Your task is to make a binary True/False judgment on whether the answers are SEMANTICALLY COMPATIBLE.

Query:
{{query}}

Expected Answer:
{{expected_answer}}
{{aliases_block}}
Generated Answer:
{{generated_answer}}

CRITICAL INSTRUCTIONS:
1. FIRST, perform a simple VERBATIM TEXT COMPARISON:
   - If the generated answer is IDENTICAL (exact same text) to EITHER the expected answer OR ANY of the answer aliases, your judgment MUST be TRUE
   - If not identical to any of them, proceed to semantic comparison

2. For SEMANTIC COMPARISON, use these MANDATORY RULES:
   - Judge "True" if the generated answer matches the SEMANTIC MEANING of EITHER the expected answer OR ANY of the answer aliases
   - Judge "True" WHENEVER the general meaning or core concept is the same as either the expected answer or any alias
   - Judge "True" if one answer is GENERAL and one is SPECIFIC about the same thing
   - Judge "True" if one answer names a CATEGORY (e.g., "missionaries") and the other provides SPECIFIC INSTANCES of that category (e.g., "Augustine was sent by Pope Gregory")
   - Judge "True" if one answer gives a BRIEF fact and the other ELABORATES with more details
   - Judge "True" if one answer is more detailed but does NOT contradict the other
   - Judge "False" ONLY if the answers directly CONTRADICT all of the expected answer and all aliases, or discuss ENTIRELY different topics

3. EXTREMELY IMPORTANT RULES ABOUT SPECIFICITY:
   - When one answer is general and one is specific → TRUE
   - When one uses a category term and one gives examples → TRUE
   - When one gives "who/what" and the other adds "when/where/how/why" → TRUE
   - When one gives a person's role and the other gives their name → TRUE
   - When one refers to a group and the other names individuals → TRUE

4. Always check if the specific answer is an INSTANCE or EXAMPLE of the general answer
   - If it is, the judgment MUST be TRUE regardless of how detailed the specific answer is

5. The query is provided ONLY for context - do NOT use it in your judgment

6. IMPORTANT: The generated answer should be considered TRUE if it matches EITHER the expected answer OR ANY of the answer aliases in meaning

FINAL CHECK BEFORE SUBMITTING:
- If the generated answer could reasonably be considered matching ANY of the expected answer or aliases → TRUE
- If after reading all answers, they feel like they're talking about the same basic concept → TRUE
- If you think "the generated answer is not contradicting the expected answer or any of its aliases" → TRUE

Your response MUST follow this format:
{
  "judgment": true/false,
  "explanation": "One clear sentence explaining why the answers are compatible or contradictory."
}"""


def render_judge(question, response):
    aliases = question["answer_aliases"]
    # Each alias wrapped in newlines, and the block closed with a blank line: this is what
    # the original jinja template emits, and tests/test_ecir_pipeline.py holds the shell
    # pipeline to it byte for byte.
    block = ("\nAnswer Aliases (Additional Correct Answers):\n"
             + "".join(f"\n- {alias}\n" for alias in aliases) + "\n\n") if aliases else "\n"
    return (
        JUDGE.replace("{{query}}", question["question"])
        .replace("{{expected_answer}}", question["short_answer"])
        .replace("{{aliases_block}}", block)
        .replace("{{generated_answer}}", response.choices[0].message.content)
    )

In [ ]:
def judge(pair):
    question, response = pair
    try:
        verdict = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": render_judge(question, response)}],
            temperature=0,
            max_completion_tokens=200,
        )
        return bool(json.loads(verdict.choices[0].message.content)["judgment"])
    except Exception:  # noqa: BLE001 - an unparseable verdict drops its example, not the run
        return None


with ThreadPoolExecutor(max_workers=8) as pool:
    verdicts = list(pool.map(judge, paired))

# judgment True == the answer was correct == not a hallucination, so the label is negated.
labelled = [(response, 0 if verdict else 1) for response, verdict in zip(responses, verdicts) if verdict is not None]
responses = [response for response, _ in labelled]
y = np.array([label for _, label in labelled])

print(f"{len(labelled)} labelled ({int(y.sum())} hallucinations)")
print(f"{len(verdicts) - len(labelled)} verdict(s) dropped as unparseable")

## Steps 4-5 — fit, and score on held-out answers

`trainable=True` returns an unfitted pipeline — parser, entropy reduction, logistic
regression — that takes the raw responses, so there is no feature extraction to write.

The split is stratified and happens before the fit, so what is reported below describes
answers the detector never saw. **ROC-AUC** scores the ranking, which is what governs
triage by score and what the paper reports; the **classification report** scores the
decisions at 0.5, where recall on the `hallucination` row is the fraction actually
flagged. Only the AUC carries over to another threshold.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from artefactual.scoring import wepr

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)
detector = wepr(k=K, trainable=True).fit(x_train, y_train)
print(f"fitted on {len(y_train)}, holding out {len(y_test)}")

scores = detector.predict_proba(x_test)[:, 1]
print(f"\nROC-AUC: {roc_auc_score(y_test, scores):.3f}")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

## Steps 6-7 — save the weights, and use them

The same `.skops` format the published detectors ship in, loaded through the same call. `k`
must be the value the weights were fitted at; another value raises rather than mis-shaping
the score.

Scoring `epr` instead is the same call with `epr` in place of `wepr`, on the same data —
it pools every rank into one feature rather than weighting them. WEPR beat EPR on every
row of the paper's table, which is why it is the default.

In [ ]:
path = detector.save_estimator("wepr-custom.skops")
reloaded = wepr(path, k=K)

print(f"P(hallucination) = {reloaded.predict_proba(responses[0])[0, 1]:.3f}")
print(responses[0].choices[0].message.content.strip())

## Where to go next

- **More data.** The fit is seconds and the generation is the cost, so raise
  `N_QUESTIONS` and rerun rather than economising on labels. A few hundred is a start;
  the paper's numbers come from far more.
- **At thousands of questions**, stop making one request per answer.
  [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) runs
  both LLM stages as `vllm run-batch` jobs, which is the practical way to do this at
  scale, and reads the same question pack step 1 built.
- **Another dataset.** Only step 1 changes. Training data should resemble the traffic
  being scored: the paper's numbers drop 10-20 points when a TriviaQA-trained detector
  meets WebQuestions.
- **Keep phase B's outputs.** The answers and their verdicts are the expensive part;
  everything from step 4 on can be rerun against them for free.